# 07 — PM2.5-radiative shading mechanism test (Medellín, O3)

Cross-city replication of Bogotá's `08_pm25_radiative_shading_test.ipynb` decisive test:
free PySR search (no pre-imposed functional form, `exp`/`log`/`sqrt` all available) on
raw `pm25` and `radiation_mean` predicting `o3`, to see whether an exponential-shading
relationship emerges on its own -- same question, different city/aerosol composition/
loading, per the 2026-09-10 "sequencing" note's cross-city-comparison rationale.


In [1]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "medellin"

df = pd.read_parquet(DATA_DIR / "medellin_o3_master_dataset.parquet")
df = df.dropna(subset=['o3']).reset_index(drop=True)
if 'pm25_lag1' not in df.columns:
    df['pm25_lag1'] = df['pm25'].shift(1)
print(df.shape)


(1423, 231)


## Same-day `pm25` + `radiation_mean`, free search

In [2]:
from pysr import PySRRegressor

features_same = ['pm25', 'radiation_mean']
data_same = df.dropna(subset=features_same + ['o3']).reset_index(drop=True)
X_same = data_same[features_same].to_numpy()
y_same = data_same['o3'].to_numpy()
print(data_same.shape)

sr_same = PySRRegressor(
    niterations=150,
    binary_operators=["+", "-", "*", "/"],
    unary_operators=["exp", "log", "sqrt"],
    model_selection="best",
    maxsize=20,
    populations=24,
)
sr_same.fit(X_same, y_same, variable_names=features_same)
print(sr_same)


Compiling Julia backend...


(1421, 231)



Expressions evaluated per second: 3.810e+04
Progress: 61 / 150 total iterations (41.111%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.066e+02  0.000e+00  y = 55.534
3           1.933e+02  3.327e-02  y = pm25 + 37.808
5           1.879e+02  1.431e-02  y = (pm25 + 66.062) * 0.66252
6           1.824e+02  2.983e-02  y = (sqrt(radiation_mean) * 2.5664) + pm25
7           1.760e+02  3.543e-02  y = ((pm25 * 0.0027953) * radiation_mean) + 45.037
9           1.759e+02  2.628e-04  y = ((pm25 - -37.458) + (radiation_mean * 0.12419)) * 0.67...
                                      665
11          1.739e+02  5.777e-03  y = sqrt(radiation_mean) - (((-139.47 / pm25) - pm25) - sq...
                                      rt(radiation_mean))
13          1.737e+02  4.387e-04  y = sqrt(radiati

[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


PySRRegressor.equations_ = [
	    pick     score                                           equation  \
	0         0.000000                                          55.534187   
	1         0.033276                                   pm25 + 37.808132   
	2         0.014316                     (pm25 * 0.6625186) + 43.787304   
	3         0.030212             sqrt(6.687031 * radiation_mean) + pm25   
	4   >>>>  0.037512  ((radiation_mean * pm25) * 0.0031956863) + 43....   
	5         0.002603  ((pm25 - -4.679882) * (0.0029075232 * radiatio...   
	6         0.003745  (1.9464655 * sqrt(radiation_mean)) - ((-156.53...   
	7         0.000720  sqrt(radiation_mean) - (((-139.47084 / pm25) -...   
	8         0.000488  sqrt(radiation_mean - pm25) - (((-146.8954 / p...   
	9         0.000217  sqrt(radiation_mean - pm25) - (((158.01054 / (...   
	10        0.000604  sqrt(radiation_mean - pm25) - (((((radiation_m...   
	
	         loss  complexity  
	0   206.64864           1  
	1   193.34354         

In [3]:
pareto_same = sr_same.equations_[['complexity', 'loss', 'equation']]
pareto_same['r2'] = pareto_same.apply(
    lambda row: r2_score(y_same, sr_same.predict(X_same, index=row.name)), axis=1
)
pareto_same


,complexity,loss,equation,r2
0,1,206.64864,55.534187,-2.447810e-07
1,3,193.34354,pm25 + 37.808132,6.438498e-02
2,5,187.88612,(pm25 * 0.6625186) + 43.787304,9.079410e-02
3,6,182.29451,sqrt(6.687031 * radiation_mean) + pm25,1.178528e-01
4,7,175.58293,((radiation_mean * pm25) * 0.0031956863) + 43....,1.503309e-01
5,9,174.67131,((pm25 - -4.679882) * (0.0029075232 * radiatio...,1.547423e-01
6,10,174.01840,(1.9464655 * sqrt(radiation_mean)) - ((-156.53...,1.579020e-01
7,11,173.89322,sqrt(radiation_mean) - (((-139.47084 / pm25) -...,1.585076e-01
8,13,173.72346,sqrt(radiation_mean - pm25) - (((-146.8954 / p...,1.593291e-01
9,15,173.64825,sqrt(radiation_mean - pm25) - (((158.01054 / (...,1.596930e-01


In [4]:
baseline_r2 = r2_score(y_same, np.polyval(np.polyfit(data_same['radiation_mean'], y_same, 1), data_same['radiation_mean']))
print(f"radiation_mean-only linear baseline R2: {baseline_r2:.3f}")
print()
print("Best (highest-R2) equation:")
print(pareto_same.loc[pareto_same['r2'].idxmax()])


radiation_mean-only linear baseline R2: 0.069

Best (highest-R2) equation:
complexity                                                   17
loss                                                  173.43874
equation      sqrt(radiation_mean - pm25) - (((((radiation_m...
r2                                                     0.160707
Name: 10, dtype: object


## Save

In [5]:
pareto_same.to_csv(DATA_DIR / "medellin_o3_pm25_shading_test_pareto.csv", index=False)
print("Saved")


Saved
